# Solar Filament V17 — New YOLO models, native tiles, multiscale search

**Aim:** Advance beyond the V16 public 0.33 submission by diagnosing the loss of small instances in whole-image 1024 inference. This experiment uses **newly trained V16 YOLO11m/s weights**, not the original 0.34 model or its submission.

### What V16 actually showed
- V16 selected YOLO11m-seg: local pooled instance PQ **0.389926** on the V16 tune split; public **0.33**.
- V16 used only a full-image `imgsz=1024` pass, `confidence=0.30`, `minimum_area=200`, and emitted **1,066** instances for 180 test images.
- V16 semantic models reported patch Dice up to ~0.743, but semantic instance PQ < YOLO11m.
- **Hypothesis:** Native image crops/tiling and modest cross-model fusion can recover fine filaments without a new lengthy training cycle.

**Requirements:** Attach the competition MAGFiLO dataset and **FULL saved V16 Kaggle output containing `models/yolo11m_seg_best.pt` and (ideally) `models/yolo11s_seg_best.pt`**. The `SOLAR_V16_REPORTS_AND_SUBMISSION.zip` contains reports/CSV only; **it does not contain the weights**. Do not try to run V17 with only the report ZIP.

**Validation limitations:** Tune set was used by the V16 training pipeline; nested selection on this same set is not a pristine new independent test. Local PQ is a proxy until verified with the official competition scorer. Historical audit is already inspected and is diagnostic only. No hidden test labels are accessed.

Run `MODE="ALL"` in a fresh Kaggle T4 kernel. Save the entire `solar_v17_native_tile` output as a Kaggle dataset and attach it to subsequent reruns.

In [ ]:
from pathlib import Path
MODE="ALL"                           # ALL | TUNE | SUBMIT
DATA_ROOT=Path("/kaggle/input/datasets/lighittha/solar3k/MAGFiLO_1.0_Kaggle_2026")
OUT=Path("/kaggle/working/solar_v17_native_tile")
V16_ROOT=None                       # autodetect saved V16 dataset with model weights
PREVIOUS_V17_ROOT=None              # optional earlier saved V17 output, to resume caches
ENABLE_YOLO11S=True
ENABLE_GLOBAL_1536=True
ENABLE_TILE_1024=True
ENABLE_TILE_1280=True
RUN_HISTORICAL_AUDIT=False          # already inspected; diagnostic only
RAW_CONF=0.08
RAW_IOU=0.72
MAX_DET=120
MODEL_IMGSZ=1024
SEED=2026
MIN_GAIN_WORTH_SUBMITTING=0.012     # over local V16 0.38993; recommendation only, never forces rollback
SAVE_CSV_EVEN_IF_NOT_BETTER=True   # CSV represents genuinely NEW V17 inference, not V7 fallback
MAX_SESSION_HOURS=11.5
RESERVE_HOURS=1.0
assert MODE in ("ALL","TUNE","SUBMIT")
for d in ["raw","reports","submissions","diagnostics"]:
    (OUT/d).mkdir(parents=True,exist_ok=True)
print("V17 config:",MODE,"/ outputs:",OUT)

In [ ]:
import os,sys,time,random,hashlib,json,math,gzip,shutil,subprocess,importlib.util,importlib.metadata,gc
from collections import defaultdict
import numpy as np,pandas as pd,cv2
from tqdm.auto import tqdm
from scipy.optimize import linear_sum_assignment
from IPython.display import display
def need(mod,pkg=None):
    if importlib.util.find_spec(mod) is None:
        subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg or mod])
need("pycocotools")
need("ultralytics")
from pycocotools import mask as coco
from ultralytics import YOLO
import ultralytics,torch
assert torch.cuda.is_available(),"Enable Kaggle GPU T4"
cv2.setNumThreads(0)
random.seed(SEED);np.random.seed(SEED)
torch.manual_seed(SEED)
H=W=2048
START=time.monotonic()
DEADLINE=START+MAX_SESSION_HOURS*3600
def remaining():return (DEADLINE-time.monotonic())/3600
def sha(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for b in iter(lambda:f.read(1<<20),b""):h.update(b)
    return h.hexdigest()
def write_json(p,thing):
    p=Path(p);p.parent.mkdir(parents=True,exist_ok=True)
    temp=p.with_name(p.name+".tmp")
    temp.write_text(json.dumps(thing,indent=2,default=str))
    temp.replace(p)
def as_rle(v):
    if isinstance(v,str):return {"size":[H,W],"counts":v.encode("ascii")}
    d=dict(v)
    if isinstance(d.get("counts"),str):d["counts"]=d["counts"].encode("ascii")
    return d
def encode(mask):return coco.encode(np.asfortranarray(np.asarray(mask,dtype=np.uint8)))
def ascii_counts(r):
    v=r["counts"];return v.decode("ascii") if isinstance(v,bytes) else str(v)
ENV={"python":sys.version,"torch":torch.__version__,"ultralytics":ultralytics.__version__,
     "gpu":torch.cuda.get_device_name(0),"seed":SEED}
write_json(OUT/"reports"/"environment.json",ENV)
print(ENV)

In [ ]:
# Locate official competition train/test, original split and NEW V16 trained weights.
ANAME="MAGFiLO_1.0_Annotations_kaggle2026_train.json"
if not (DATA_ROOT/"train"/ANAME).exists():
    opts=list(Path("/kaggle/input").rglob(ANAME))
    if len(opts)!=1:raise FileNotFoundError("Attach the official MAGFiLO competition dataset.")
    DATA_ROOT=opts[0].parent.parent
ANNOT=DATA_ROOT/"train"/ANAME
TRAIN={p.stem:p for p in (DATA_ROOT/"train"/"train_images").iterdir() if p.suffix.lower() in (".jpg",".jpeg",".png")}
TEST={p.stem:p for p in (DATA_ROOT/"test"/"test_images").iterdir() if p.suffix.lower() in (".jpg",".jpeg",".png")}
assert len(TRAIN)==707 and len(TEST)==180,(len(TRAIN),len(TEST))
DOC=json.loads(ANNOT.read_text())
ANN=defaultdict(list)
for a in DOC["annotations"]:ANN[str(a["image_id"])].append(a)
VERSIONS=defaultdict(list)
for item in DOC["images"]:
    stem=Path(item["file_name"]).stem
    if stem in TRAIN:VERSIONS[stem].append((str(item["id"]),ANN[str(item["id"])]))
assert set(VERSIONS)==set(TRAIN)
split_files=list(Path("/kaggle/input").rglob("split_manifest.json"))
expected="f9c906951de50e5245c7411ceba5ebd093eff2368f9719c628dc8d1d7e52361c"
split=next((p for p in split_files if sha(p)==expected),None)
if split is None:
    raise FileNotFoundError("Attach V16/V4 split_manifest.json or the V16 REPORTS zip as extracted input.")
SPLIT=json.loads(split.read_text())
assert [len(SPLIT[k]) for k in ("train","tune","audit")]==[513,97,97]
assert set().union(*(set(x) for x in SPLIT.values()))==set(TRAIN)
assert not ({s[:8] for s in SPLIT["train"]}&{s[:8] for s in SPLIT["tune"]})
assert not ({s[:8] for s in SPLIT["audit"]}&{s[:8] for s in SPLIT["tune"]})
# Exact trained V16 filenames only (never use prior V7/V13 checkpoint).
WEIGHTS={}
for name in ["yolo11m_seg"]+(["yolo11s_seg"] if ENABLE_YOLO11S else []):
    if V16_ROOT is not None:
        opts=list(Path(V16_ROOT).rglob(name+"_best.pt"))
    else:
        opts=list(Path("/kaggle/input").rglob(name+"_best.pt"))
    if len(opts)!=1:
        raise FileNotFoundError(f"V17 needs the NEW V16 model weights {name+'_best.pt'}; found {len(opts)}. "
            "Attach the FULL V16 Kaggle output (not reports-only ZIP).")
    WEIGHTS[name]=opts[0]
IDENTITY={n:sha(p) for n,p in WEIGHTS.items()}
print("V16 trained models:",{n:str(p) for n,p in WEIGHTS.items()})
write_json(OUT/"reports"/"data_and_weights.json",{"data_root":str(DATA_ROOT),"split_sha256":sha(split),
  "weights":{n:{"path":str(p),"sha256":IDENTITY[n]} for n,p in WEIGHTS.items()},
  "competition_annotations_only":True,"test_labels_used":False})

In [ ]:
# Official-train-annotation-only local PQ (diagnostic proxy, NOT confirmed organizer parity).
def annotation_rle(a):
    seg=a.get("segmentation")
    if seg is None or seg==[]:return None
    if isinstance(seg,dict):
        if isinstance(seg.get("counts"),list):return coco.frPyObjects(seg,H,W)
        return as_rle(seg)
    if len(seg) and isinstance(seg[0],(float,int)):seg=[seg]
    poly=[p for p in seg if len(p)>=6 and len(p)%2==0]
    if not poly:return None
    raw=coco.frPyObjects(poly,H,W)
    return coco.merge(raw) if isinstance(raw,list) else raw
TRUTH={}
def gt(s):
    if s not in TRUTH:
        TRUTH[s]=[[r for a in items if (r:=annotation_rle(a)) is not None]
                   for _,items in VERSIONS[s]]
    return TRUTH[s]
def counts(g,p):
    if not g:return np.asarray([0.,0.,float(len(p)),0.])
    if not p:return np.asarray([0.,0.,0.,float(len(g))])
    overlap=np.asarray(coco.iou(g,p,[0]*len(p)),float)
    ri,ci=linear_sum_assignment(-overlap)
    valid=overlap[ri,ci]>.5
    true_positive=int(valid.sum())
    return np.asarray([float(overlap[ri[valid],ci[valid]].sum()),
        float(true_positive),float(len(p)-true_positive),float(len(g)-true_positive)])
def pooled(s,p):
    total=np.zeros(4,float)
    for truth in gt(s):total+=counts(truth,p)
    return total
def pq(x):
    den=float(x[1]+.5*x[2]+.5*x[3])
    return float(x[0]/den) if den else 0.
def summarize(x):
    x=np.asarray(x,float);den=x[1]+.5*(x[2]+x[3])
    return {"PQ":pq(x),"SQ":float(x[0]/x[1]) if x[1] else 0.,
        "RQ":float(x[1]/den) if den else 0.,
        "TP":int(x[1]),"FP":int(x[2]),"FN":int(x[3])}
unit=np.zeros((H,W),np.uint8);unit[40:100,40:120]=1
r=encode(unit)
assert abs(pq(counts([r],[r]))-1)<1e-9
assert pq(counts([r],[]))==0
print("Local one-to-one PQ diagnostic initialized. Official metric parity UNVERIFIED.")

In [ ]:
# Native-resolution YOLO proposal cache. Deterministic, source-preserving; cached RLE keyed to weight SHA/version.
MODEL_CACHE={}
VIEW_ORDER=["g1024"]+ (["g1536"] if ENABLE_GLOBAL_1536 else []) + (
    ["t1280"] if ENABLE_TILE_1280 else []) + (["t1024"] if ENABLE_TILE_1024 else [])
def get_engine(name):
    if name not in MODEL_CACHE:
        MODEL_CACHE.clear()
        gc.collect();torch.cuda.empty_cache()
        MODEL_CACHE[name]=YOLO(str(WEIGHTS[name]))
    return MODEL_CACHE[name]
def windows(view):
    if view[0]=="g":return [(0,0,2048)]
    if view=="t1280":
        return [(x,y,1280) for y in (0,768) for x in (0,768)]
    if view=="t1024":
        return [(x,y,1024) for y in (0,512,1024) for x in (0,512,1024)]
    raise ValueError(view)
RAW_ID=hashlib.sha256(json.dumps({"weights":IDENTITY,"ultralytics":ultralytics.__version__,
    "raw_conf":RAW_CONF,"raw_iou":RAW_IOU,"max_det":MAX_DET,
    "views":VIEW_ORDER,"retina_masks":True},sort_keys=True).encode()).hexdigest()
def raw_file(model,s,view):return OUT/"raw"/model/s/(view+".json.gz")
def archived_raw_file(model,s,view):
    if not PREVIOUS_V17_ROOT:return None
    return Path(PREVIOUS_V17_ROOT)/"raw"/model/s/(view+".json.gz")
def read_gz(p):
    with gzip.open(p,"rt") as f:return json.load(f)
def write_gz(p,data):
    p.parent.mkdir(parents=True,exist_ok=True)
    temp=p.with_name(p.name+".tmp")
    with gzip.open(temp,"wt",compresslevel=4) as f:json.dump(data,f,separators=(",",":"))
    temp.replace(p)
@torch.inference_mode()
def proposals(model,s,view):
    filepath=raw_file(model,s,view)
    for f in (filepath,archived_raw_file(model,s,view)):
        if f and f.exists():
            obj=read_gz(f)
            if obj.get("identity")==RAW_ID and obj.get("stem")==s and obj.get("view")==view:
                return obj["proposals"]
    path=(TRAIN[s] if s in TRAIN else TEST[s])
    img=cv2.imread(str(path),cv2.IMREAD_GRAYSCALE)
    if img is None or img.shape!=(H,W):raise RuntimeError(f"Cannot read 2048 image {path}")
    output=[]
    engine=get_engine(model)
    for x,y,side in windows(view):
        crop=np.ascontiguousarray(img[y:y+side,x:x+side])
        inp=cv2.cvtColor(crop,cv2.COLOR_GRAY2BGR)
        size=1024 if view.startswith("t") else int(view[1:])
        result=engine.predict(source=inp,imgsz=size,conf=RAW_CONF,iou=RAW_IOU,
            retina_masks=True,agnostic_nms=True,max_det=MAX_DET,device=0,verbose=False)[0]
        if result.masks is None or result.boxes is None:continue
        mm=result.masks.data.cpu().numpy()
        conf=result.boxes.conf.cpu().numpy()
        for a,score in zip(mm,conf):
            local=(a>.5).astype(np.uint8)
            if local.shape!=(side,side):
                local=cv2.resize(local,(side,side),interpolation=cv2.INTER_NEAREST)
            if int(local.sum())<100:continue
            if side==2048:full=local
            else:
                full=np.zeros((H,W),np.uint8)
                full[y:y+side,x:x+side]=local
            truncated=False
            if side<2048:
                ys,xs=np.where(local)
                if len(xs):
                    truncated=bool((x>0 and xs.min()<10) or (x+side<W and xs.max()>=side-10)
                        or (y>0 and ys.min()<10) or (y+side<H and ys.max()>=side-10))
            rr=encode(full)
            output.append({"counts":ascii_counts(rr),"area":int(coco.area(rr)),
                "score":float(score),"origin":view,"model":model,"truncated":truncated})
    record={"identity":RAW_ID,"stem":s,"view":view,"proposals":output}
    write_gz(filepath,record)
    return output
def need_cache(stems,models,views):
    tasks=[(m,s,v) for m in models for s in stems for v in views]
    for m,s,v in tqdm(tasks,desc="V17 exact multiscale proposal caches"):
        if remaining()<RESERVE_HOURS+.1:raise RuntimeError(
            "Kaggle session budget nearly exhausted. Save V17 output and attach next session to resume.")
        proposals(m,s,v)
    print("Completed raw caches:",len(tasks),"for",len(stems),"images.")
print("Active views:",VIEW_ORDER)

In [ ]:
# Candidate generator and conflict-aware instance selection. No old V7/V13 recipe is used.
def select(proposals_by_source,cfg):
    available=[]
    for (model,view),entries in proposals_by_source.items():
        if model not in cfg["models"] or view not in cfg["views"]:continue
        for p in entries:
            area=int(p["area"]);score=float(p["score"])
            thr=cfg["tile_conf"] if view.startswith("t") else cfg["global_conf"]
            if score<thr or area<cfg["min_area"]:continue
            if cfg["seam_guard"] and view.startswith("t") and p["truncated"]:continue
            rank=score + (cfg["tile_bonus"] if view.startswith("t") else 0.)
            if model=="yolo11m_seg":rank+=cfg.get("m_bonus",0.)
            available.append((rank,area,p))
    available.sort(key=lambda q:(-q[0],-q[1]))
    output=[];accepted_raw=[];occupied=np.zeros((H,W),np.uint8)
    for rank,area,p in available:
        rr=as_rle(p["counts"])
        if accepted_raw:
            ov=np.asarray(coco.iou([rr],accepted_raw,[0]*len(accepted_raw)))[0]
            if np.any(ov>=cfg["nms_iou"]):continue
        rawmask=coco.decode(rr).astype(np.uint8)
        free=rawmask&(occupied==0)
        num=int(free.sum())
        if num<cfg["min_area"] or num/float(max(1,area))<cfg["min_free"]:continue
        out=encode(free)
        output.append(out);accepted_raw.append(rr);occupied|=free
        if len(output)>=120:break
    return output
def configs():
    presets=[
        ("m_global",["yolo11m_seg"],["g1024"]),
        ("m_global_hi",["yolo11m_seg"],["g1536"]),
        ("m_native_1280",["yolo11m_seg"],["t1280"]),
        ("m_native_1024",["yolo11m_seg"],["t1024"]),
        ("m_global_and_1280",["yolo11m_seg"],["g1024","t1280"]),
        ("m_global_hi_and_1280",["yolo11m_seg"],["g1536","t1280"]),
        ("m_global_and_1024",["yolo11m_seg"],["g1024","t1024"]),
        ("m_global_hi_and_1024",["yolo11m_seg"],["g1536","t1024"]),
        ("m_multi_native",["yolo11m_seg"],["g1024","t1280","t1024"]),
        ("m_all",["yolo11m_seg"],["g1024","g1536","t1280","t1024"]),
        ("s_global",["yolo11s_seg"],["g1024"]),
        ("s_native_1280",["yolo11s_seg"],["t1280"]),
        ("s_global_and_1024",["yolo11s_seg"],["g1024","t1024"]),
        ("ms_global",["yolo11m_seg","yolo11s_seg"],["g1024"]),
        ("ms_global_native_1280",["yolo11m_seg","yolo11s_seg"],["g1024","t1280"]),
        ("ms_global_native_1024",["yolo11m_seg","yolo11s_seg"],["g1024","t1024"]),
        ("ms_all",["yolo11m_seg","yolo11s_seg"],["g1024","t1280","t1024"])
    ]
    candidates=[]
    for name,models,views in presets:
        if not all(m in WEIGHTS for m in models) or not all(v in VIEW_ORDER for v in views):continue
        for global_conf,tile_conf,min_area,seam in [
            (.22,.34,200,False),(.30,.40,250,False),(.18,.28,200,True),
            (.25,.36,350,True)]:
            candidates.append(dict(name=f"{name}_gc{int(global_conf*100)}_tc{int(tile_conf*100)}_a{min_area}_s{int(seam)}",
                models=models,views=views,global_conf=global_conf,tile_conf=tile_conf,
                min_area=min_area,seam_guard=seam,nms_iou=.50,
                tile_bonus=.08,min_free=.45,m_bonus=.04 if len(models)>1 else 0.))
    return candidates
CONFIGS=configs()
MODEL_NAMES=list(WEIGHTS)
print("V17 distinct strategies:",len(CONFIGS))

In [ ]:
# V17 validation: acquisition-day screen and full 97-scene measurement.
# The 97-scene set was also used by V16 for checkpoint selection;
# report PQ as exploratory model-selection performance, NOT a pristine test.
STEMS=sorted(SPLIT["tune"])
DAY_GROUPS=defaultdict(list)
for s in STEMS:DAY_GROUPS[s[:8]].append(s)
days=sorted(DAY_GROUPS)
random.Random(SEED+17).shuffle(days)
SCREEN=[]
for day in days:
    if len(SCREEN)>=32:break
    SCREEN.extend(DAY_GROUPS[day])
SCREEN=sorted(SCREEN)
OTHER=sorted(set(STEMS)-set(SCREEN))
assert not ({s[:8] for s in SCREEN}&{s[:8] for s in OTHER})
print("Screen images:",len(SCREEN),"remaining:",len(OTHER))
if MODE in ("ALL","TUNE"):
    need_cache(STEMS,MODEL_NAMES,VIEW_ORDER)
    # Load caches one image at a time; the 2048 RLE encodings are memory-efficient.
    def get_image_raw(stem):
        return {(m,v):proposals(m,stem,v) for m in MODEL_NAMES for v in VIEW_ORDER}
    candidates={i:np.zeros(4) for i in range(len(CONFIGS))}
    for s in tqdm(SCREEN,desc="Native-tile first-pass recipes"):
        raw=get_image_raw(s)
        for i,cfg in enumerate(CONFIGS):
            candidates[i]+=pooled(s,select(raw,cfg))
    first=sorted(candidates,key=lambda i:pq(candidates[i]),reverse=True)
    selected_ids=first[:min(12,len(first))]
    if not selected_ids:raise RuntimeError("No V17 candidates")
    write_json(OUT/"reports"/"screen.json",{"top":[{"recipe":CONFIGS[i]["name"],**summarize(candidates[i])} for i in first[:30]],
        "screen_n":len(SCREEN),"selected":selected_ids})
    scores={i:np.zeros(4) for i in selected_ids}
    day_vectors={i:defaultdict(lambda:np.zeros(4)) for i in selected_ids}
    per_image=[]
    for s in tqdm(STEMS,desc="V17 full 97-image pooled PQ"):
        raw=get_image_raw(s)
        for i in selected_ids:
            rr=select(raw,CONFIGS[i]);co=pooled(s,rr)
            scores[i]+=co;day_vectors[i][s[:8]]+=co
            per_image.append({"stem":s,"day":s[:8],"recipe":CONFIGS[i]["name"],
                "instances":len(rr),**summarize(co)})
    rank=sorted(selected_ids,key=lambda i:pq(scores[i]),reverse=True)
    board=pd.DataFrame([{"rank":j+1,"recipe":CONFIGS[i]["name"],
        **summarize(scores[i]),"screen_PQ":pq(candidates[i]),
        "model_names":"+".join(CONFIGS[i]["models"]),
        "views":"+".join(CONFIGS[i]["views"])} for j,i in enumerate(rank)])
    board.to_csv(OUT/"reports"/"v17_validation_leaderboard.csv",index=False)
    pd.DataFrame(per_image).to_csv(OUT/"reports"/"v17_per_image.csv",index=False)
    display(board.head(15))
    chosen=CONFIGS[rank[0]]
    # Holdout day groups are used for stability diagnostics, not separate model selection.
    fold=[[],[]]
    for s in STEMS:
        group=s[:8]
        if group not in {x[:8] for x in fold[0]} and group not in {x[:8] for x in fold[1]}:
            k=0 if len(fold[0])<=len(fold[1]) else 1
        else:
            k=0 if group in {x[:8] for x in fold[0]} else 1
        fold[k].append(s)
    valframe=pd.DataFrame(per_image)
    stability=[]
    for k,group in enumerate(fold):
        f=valframe[(valframe["recipe"]==chosen["name"])&valframe["stem"].isin(group)]
        # Per-image pooled sums: use numerator sum_iou = PQ * (TP+.5*(FP+FN))
        n=float((f["PQ"]*(f["TP"]+.5*(f["FP"]+f["FN"]))).sum())
        total=np.array([n,f.TP.sum(),f.FP.sum(),f.FN.sum()])
        stability.append({"part":k,"images":len(group),**summarize(total)})
    write_json(OUT/"reports"/"selection.json",{
        "selected_strategy":chosen,"local_pq":float(pq(scores[rank[0]])),
        "v16_local_yolo11m_pq":0.3899255103900492,
        "exploratory_gain_over_v16":float(pq(scores[rank[0]])-0.3899255103900492),
        "recommend_submit":bool(pq(scores[rank[0]])-0.3899255103900492>=MIN_GAIN_WORTH_SUBMITTING),
        "group_stability":stability,"metric_parity_confirmed":False,
        "validation_tune_already_used_in_v16_model_selection":True,
        "hidden_test_labels_used":False,"raw_identity":RAW_ID})
    print("NEW V17 SELECTED:",chosen["name"],"local PQ",pq(scores[rank[0]]))
else:
    files=[OUT/"reports"/"selection.json"]
    if PREVIOUS_V17_ROOT:files.append(Path(PREVIOUS_V17_ROOT)/"reports"/"selection.json")
    decision=next((json.loads(p.read_text()) for p in files if p.is_file()),None)
    if decision is None:raise FileNotFoundError("Run TUNE first, or attach previous V17 output with selection.json")
    assert decision["raw_identity"]==RAW_ID
    chosen=decision["selected_strategy"]
    print("Recovered V17 strategy:",chosen["name"])

In [ ]:
# Optional old audit — diagnostic only, already inspected in historical work.
if RUN_HISTORICAL_AUDIT and MODE in ("ALL","TUNE"):
    old=sorted(SPLIT["audit"])
    relevant_models=chosen["models"];relevant_views=chosen["views"]
    need_cache(old,relevant_models,relevant_views)
    total=np.zeros(4)
    for s in tqdm(old,desc="Previously inspected audit; NOT independent"):
        raw={(m,v):proposals(m,s,v) for m in relevant_models for v in relevant_views}
        total+=pooled(s,select(raw,chosen))
    write_json(OUT/"reports"/"old_audit_diagnostic.json",{
        **summarize(total),"images":len(old),"historical_audit_is_pristine":False})

In [ ]:
# Export an actually-new V17 multiscale/new-model CSV. No fallback to 0.34/0.33 old CSV.
if MODE in ("ALL","SUBMIT"):
    decision_file=OUT/"reports"/"selection.json"
    if not decision_file.exists() and PREVIOUS_V17_ROOT:
        decision_file=Path(PREVIOUS_V17_ROOT)/"reports"/"selection.json"
    if not decision_file.is_file():raise FileNotFoundError("No selected V17 policy")
    decision=json.loads(decision_file.read_text())
    assert decision["raw_identity"]==RAW_ID
    selected=decision["selected_strategy"]
    if not decision["recommend_submit"]:
        print("CAUTION: V17 did not achieve +",MIN_GAIN_WORTH_SUBMITTING,
              "local PQ vs V16. Do not spend a public submission blindly.")
    need_cache(sorted(TEST),selected["models"],selected["views"])
    rows=[];counts_table=[]
    for stem in tqdm(sorted(TEST),desc="New multiscale V17 test CSV"):
        raw={(m,v):proposals(m,stem,v) for m in selected["models"] for v in selected["views"]}
        rr=select(raw,selected)
        counts_table.append({"stem":stem,"instances":len(rr)})
        for j,p in enumerate(rr,1):
            assert int(coco.area(p))>0
            rows.append((f"{stem}_{j}",ascii_counts(p)))
    csv=pd.DataFrame(rows,columns=["filament_id","segmentation_rle"])
    assert len(csv)>=200 and csv.filament_id.is_unique
    assert csv.segmentation_rle.astype(bool).all()
    assert set(csv.filament_id.str.rsplit("_",n=1).str[0]).issubset(set(TEST))
    # Validate every RLE's dimensions and positive area.
    for c in csv.segmentation_rle:
        r=as_rle(c)
        assert r["size"]==[H,W] and int(coco.area(r))>0
    dest=OUT/"submissions"/"V17_NEW_YOLO_MULTISCALE_submission.csv"
    csv.to_csv(dest,index=False)
    pd.DataFrame(counts_table).to_csv(OUT/"reports"/"test_instance_counts.csv",index=False)
    write_json(OUT/"reports"/"submission_report.json",{
        "csv":str(dest),"rows":len(csv),"test_images":len(TEST),
        "nonempty_images":int(sum(c["instances"]>0 for c in counts_table)),
        "sha256":sha(dest),"selected":selected,"old_v16_submission_reused":False,
        "v7_submission_reused":False,"test_labels_used":False})
    if SAVE_CSV_EVEN_IF_NOT_BETTER:
        shutil.copy2(dest,"/kaggle/working/submission.csv")
    print("V17 NEW CSV:",dest,"rows:",len(csv),"sha256:",sha(dest))
print("V17 outputs at:",OUT)

## Interpreting your V17 result

`reports/v17_validation_leaderboard.csv` compares independent genuinely different inference strategies from newly trained V16 YOLO11m/s weights. `reports/selection.json` contains the chosen strategy, local pooled PQ and stability diagnostics. `submissions/V17_NEW_YOLO_MULTISCALE_submission.csv` is **not** an old baseline copy.

- **If +0.012 or more local PQ improvement** and the selection is stable, consider a single leaderboard submission after inspecting masks and comparing with the competition's **official** self-evaluation metric.
- **If not**, avoid wasting submissions. The next meaningful step is **native-tile training** (train YOLO on 1024×1024 original-resolution labeled crops) rather than another global YOLO fine-tune.
- `RAW_ID` ties raw caches to exact weights, model version, conf, IoU, and views, so they can be reused safely across sessions.
- Attach the **full V16 output** with model checkpoints and **full V17 output** with caches for continuity. This reports-only ZIP alone does not contain model weights.